## RDS security & RDS Proxy

RDS security is four layers. **Network** — instances live in a VPC, normally **private subnets**, with a security group allowing the engine's port (5432/3306/1521) **from the app tier's SG only**; there's no production reason for a public RDS. **Encryption at rest** is KMS AES-256 and **must be enabled at creation** (to encrypt an existing instance: snapshot → copy-with-encryption → restore → repoint); replicas of an encrypted primary are auto-encrypted. **Encryption in transit** is SSL/TLS enforced via parameter group, validated against RDS's regional CA bundle. **IAM database authentication** lets the app use a short-lived token instead of a password — no stored DB credential.

**RDS Proxy** solves a different problem: relational DBs cap concurrent connections (each is expensive — TLS, auth, memory), and while long-running app servers pool connections naturally, **Lambda doesn't** — every invocation opens its own, so a few thousand concurrent invocations exhaust the DB. RDS Proxy is a **managed connection pool** between app and database, multiplexing thousands of app connections into a small pool of long-lived backend ones. Three properties: it **handles failover** (reconnects to the new primary in seconds while the app's connection to the proxy stays open), enforces **IAM auth + Secrets Manager** credentials, and is the **standard part of any Lambda-plus-relational** stack.